### STEP 1 LOAD MODEL & TOKENIZER

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True
)

### STEP 2 ADD LORA ADAPTERS 

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target = ["q_proj", "k_proj", "v_proj", "o_proj", 
                "gate_proj", "up_proj", "down_proj"]
    lora_alpha = 32,
    lora_dropout = 0,
    random_state = 3044
)

### STEP 3 FORMAT DATASET

In [ ]:
from datasets import load_dataset
from unsloth import get_chat_template


# ATTACH LLAMA 3.2 TEMPLATE TO TOKENIZER
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.2"
)

# LOAD LOCAL MARWARI DATASET
dataset = load_dataset("json", data_files = "marwari_sft.json", split = 'train')


# DEFINING THE FORMATTING FUNCTION
def format_prompts(sample):

    texts = []

    for inst, out in zip(sample['instructions'], sample['output']):
        message = [{
            "role" : "user",
            "content": inst },
            {"role" : "assistant",
            "content": out }]
        
        text.append(tokenizer.apply_chat_template(message), tokenize = False)
    
    return {"text" : texts}

dataset = dataset.map(format_prompts, batched = True)


### STEP 4 TRANING USING SFT

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset= dataset,
    dataset_text_field = 'text',
    max_seq_length = max_seq_length,

    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        learning_rate = 2e-4,
        logging_steps = 10,
        max_steps = 60,
        optim = "adamw_8bit",
        output_dir = "outputs",
    ),

    
)